# Resumen de lo realizado en el pipeline (EDA + Feature Engineering + Modelado)

En este proyecto construimos un flujo completo de riesgo de crédito sobre Home Credit Default Risk usando PySpark/SparkSQL en AWS Glue, con almacenamiento en S3 y tablas en Glue Catalog.

## 1) Integración de fuentes y contexto

Se trabajó con las tablas principales del ecosistema Home Credit:
- `application_train`
- `previous_application`
- `installments_payments`
- `bureau`
- `bureau_balance`
- `pos_cash_balance`
- `credit_card_balance`

La estrategia fue leer desde Glue Catalog y usar S3 como respaldo cuando correspondía, manteniendo procesamiento distribuido (sin depender de Pandas para volumen grande).

## 2) EDA técnico por dominios

Se hicieron análisis exploratorios por bloques de negocio:
- **Installments**: comportamiento de pago esperado vs pago real, atrasos y subpagos.
- **POS/Credit Card**: mora mensual, uso de líneas de crédito, estados contractuales.
- **Bureau/Bureau Balance**: historial externo del cliente, severidad de mora histórica, calidad de datos e inconsistencias.

Estos EDA permitieron identificar señales de riesgo, reglas de limpieza y features con mayor potencial predictivo.

## 3) Transformaciones y limpieza aplicadas

Se aplicó una lógica robusta y explicable para producción:
- normalización de nombres de columnas;
- validaciones de esquema y manejo de columnas faltantes;
- tratamiento de anomalías (por ejemplo códigos especiales en variables de días);
- creación de flags de faltantes (`*_missing_flag`);
- cálculo de ratios financieros con división segura;
- capping por percentiles para controlar outliers;
- transformaciones `log1p` en variables monetarias sesgadas;
- creación de variables derivadas temporales y de comportamiento.

## 4) Features creadas (nivel cliente)

Se consolidaron features por `sk_id_curr` en cada fuente:
- **Application**: ratios de ingreso/crédito/anualidad, edad, empleo, estructura familiar.
- **Bureau**: cantidad de créditos, activos/cerrados, deuda total, mora máxima, señales de inconsistencia.
- **Bureau Balance**: proporción de meses en mora, peor estado histórico, meses cerrados/desconocidos.
- **Previous Application**: estatus históricos (approved/refused), montos y razones de solicitud.
- **Installments**: `payment_delay`, `payment_ratio`, atraso severo, subpagos y no pagos.
- **POS Cash**: DPD promedio/máximo y estatus del contrato.
- **Credit Card**: balances, límites, DPD, pagos y `utilization_ratio`.

## 5) Construcción de dataset maestro

Se construyó un `master_dataset` mediante `LEFT JOIN` sobre `sk_id_curr`, partiendo de la base de `application_train` y anexando agregados de cada tabla satélite.

Validaciones realizadas:
- consistencia de filas vs base;
- no duplicación de cliente;
- revisión de nulos principales;
- trazabilidad de outputs generados.

## 6) Dataset listo para SparkML

Se generó versión ML-ready con:
- imputación de numéricas (mediana);
- codificación de categóricas (`StringIndexer` + `OneHotEncoder`);
- ensamblado de vector de entrada (`VectorAssembler`);
- etiqueta final `label = target`.

Resultado: dataset utilizable directamente por modelos de clasificación en SparkML.

## 7) Modelado y evaluación inicial

Se entrenó un baseline de clasificación (Random Forest) y se analizaron:
- desempeño inicial del modelo;
- importancia de variables;
- ranking rápido de features para priorizar iteraciones.

## 8) Conclusión general

El trabajo completó el ciclo: EDA -> reglas de transformación -> generación de features -> dataset maestro -> dataset ML-ready -> baseline model.

Esto deja al proyecto en estado listo para:
- tuning de hiperparámetros,
- selección fina de variables,
- comparación entre modelos (RF, GBT, Logistic Regression),
- y construcción de resultados finales para el informe académico.

In [ ]:
# ============================================================
# CHUNK 1 — Cargar dataset final de features desde Glue o S3
# ============================================================

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder.getOrCreate()


def read_from_glue_or_s3(table_name: str, s3_path: str):
    """
    Intenta leer primero desde Glue Catalog.
    Si la tabla no existe o falla la lectura, cae a Parquet en S3.
    """
    try:
        if spark.catalog.tableExists(table_name):
            print(f"Leyendo desde Glue: {table_name}")
            return spark.table(table_name)
        else:
            print(f"No existe en Glue: {table_name}. Leyendo desde S3: {s3_path}")
            return spark.read.parquet(s3_path)

    except Exception as e:
        print(f"Falló lectura desde Glue para {table_name}: {e}")
        print(f"Leyendo desde S3: {s3_path}")
        return spark.read.parquet(s3_path)


# ------------------------------------------------------------
# Ruta real del dataset final de features
# ------------------------------------------------------------

df_train = read_from_glue_or_s3(
    table_name="trusted_db.train_final_features",
    s3_path="s3://hcdr/refined/train_final_features/"
)


# ------------------------------------------------------------
# Validación inicial
# ------------------------------------------------------------

print("Datos cargados correctamente")
print("Filas:", df_train.count())
print("Columnas:", len(df_train.columns))

df_train.printSchema()

df_train.show(5, truncate=False)

In [ ]:
# ============================================================
# CHUNK 2 — Crear features SIN limpieza adicional
# ============================================================

from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, DecimalType
from pyspark.ml.feature import VectorAssembler
import pyspark.sql.functions as F

# ------------------------------------------------------------
# 1. Usar el dataset cargado en el CHUNK 1
# ------------------------------------------------------------

df_model = df_train

# ------------------------------------------------------------
# 2. Validar que exista target
# ------------------------------------------------------------

print("Distribución de target:")
df_model.groupBy("target").count().show()

# SparkML necesita una columna label numérica
# Esto NO es limpieza, solo es adaptar el nombre/formato para SparkML
df_model = df_model.withColumn(
    "label",
    F.col("target").cast("double")
)

In [ ]:
# 3. Columnas que NO deben entrar al modelo
# ------------------------------------------------------------

exclude_cols = [
    "sk_id_curr",
    "sk_id_prev",
    "sk_id_bureau",
    "target",
    "label"
]

# ------------------------------------------------------------
# 4. Seleccionar features numéricas tal como vienen
# ------------------------------------------------------------

feature_cols = []

for field in df_model.schema.fields:
    col_name = field.name

    if col_name not in exclude_cols:
        if isinstance(field.dataType, (IntegerType, LongType, DoubleType, FloatType, DecimalType)):
            feature_cols.append(col_name)

print("Cantidad de features seleccionadas:", len(feature_cols))
print("Primeras features:")
print(feature_cols[:30])

In [ ]:
# ------------------------------------------------------------
# 5. Crear columna features
# ------------------------------------------------------------

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

df_features = assembler.transform(df_model)

In [ ]:
# ============================================================
# CHUNK 3 — Dividir datos en train y test
# ============================================================

# ------------------------------------------------------------
# 1. Dividir el dataset
# ------------------------------------------------------------
# train_df: datos que el modelo usa para aprender
# test_df : datos que el modelo NO ve durante entrenamiento,
#           y se usan para evaluar qué tan bien predice

train_df, test_df = df_features.randomSplit(
    [0.8, 0.2],
    seed=42
)

# ------------------------------------------------------------
# 2. Revisar tamaños
# ------------------------------------------------------------

print("Filas train:", train_df.count())
print("Filas test :", test_df.count())

# ------------------------------------------------------------
# 3. Revisar distribución del target en train
# ------------------------------------------------------------

print("Distribución label en train:")
train_df.groupBy("label").count().show()

# ------------------------------------------------------------
# 4. Revisar distribución del target en test
# ------------------------------------------------------------

print("Distribución label en test:")
test_df.groupBy("label").count().show()

print("✅ CHUNK 3 ejecutado correctamente")
print("Dataset dividido en train y test")

In [ ]:
# ============================================================
# CHUNK 3.5 — Estrategia de pesos por desbalance de TARGET
# ============================================================

import pyspark.sql.functions as F

# ------------------------------------------------------------
# 1. Revisar distribución de clases en train
# ------------------------------------------------------------

class_counts = train_df.groupBy("label").count()

print("Distribución original en train:")
class_counts.show()

# ------------------------------------------------------------
# 2. Obtener cantidad de registros por clase
# ------------------------------------------------------------

counts_dict = {
    row["label"]: row["count"]
    for row in class_counts.collect()
}

count_0 = counts_dict.get(0.0, 0)
count_1 = counts_dict.get(1.0, 0)

total = count_0 + count_1

print("Clase 0:", count_0)
print("Clase 1:", count_1)
print("Total train:", total)

# ------------------------------------------------------------
# 3. Calcular pesos balanceados
# ------------------------------------------------------------
# Fórmula:
# peso_clase = total / (número_clases * cantidad_clase)

weight_0 = total / (2.0 * count_0)
weight_1 = total / (2.0 * count_1)

print("Peso clase 0:", weight_0)
print("Peso clase 1:", weight_1)

# ------------------------------------------------------------
# 4. Crear columna class_weight
# ------------------------------------------------------------

train_weighted_df = train_df.withColumn(
    "class_weight",
    F.when(F.col("label") == 0.0, F.lit(weight_0))
     .when(F.col("label") == 1.0, F.lit(weight_1))
     .otherwise(F.lit(1.0))
)

# ------------------------------------------------------------
# 5. Verificar resultado
# ------------------------------------------------------------

print("Train con pesos:")
train_weighted_df.groupBy("label", "class_weight").count().show()

print("✅ CHUNK 3.5 ejecutado correctamente")

In [ ]:
# ============================================================
# CHUNK 4 — Entrenar RandomForestClassifier con pesos
# ============================================================

from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=8,
    seed=42,
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8
)

rf_model = rf.fit(train_weighted_df)

print("✅ Modelo Random Forest entrenado con pesos de clase")

In [ ]:
# ============================================================
# CHUNK 5 — Generar predicciones sobre test
# ============================================================

predictions = rf_model.transform(test_df)

print("✅ Predicciones generadas sobre test")

predictions.select(
    "sk_id_curr",
    "target",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

In [ ]:
# ============================================================
# CHUNK 6 — Comparar predicción contra target
# ============================================================

import pyspark.sql.functions as F

# ------------------------------------------------------------
# 1. Ver ejemplos de predicciones vs valor real
# ------------------------------------------------------------

predictions.select(
    "sk_id_curr",
    "target",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)


# ------------------------------------------------------------
# 2. Matriz de confusión
# ------------------------------------------------------------
# label      = valor real
# prediction = valor predicho

confusion_matrix = (
    predictions
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_matrix.show()

In [ ]:
# ============================================================
# CHUNK — Métricas con porcentajes TP, TN, FP, FN
# ============================================================

import pyspark.sql.functions as F

# Valores ya calculados
tp = 3144
tn = 40012
fp = 16219
fn = 1733

total = tp + tn + fp + fn

total_real_0 = tn + fp   # todos los clientes reales NO default
total_real_1 = tp + fn   # todos los clientes reales default

# ------------------------------------------------------------
# Porcentaje sobre el total del test
# ------------------------------------------------------------

pct_tp_global = 100 * tp / total
pct_tn_global = 100 * tn / total
pct_fp_global = 100 * fp / total
pct_fn_global = 100 * fn / total

# ------------------------------------------------------------
# Porcentaje dentro de cada clase real
# ------------------------------------------------------------

pct_tn_dentro_real_0 = 100 * tn / total_real_0
pct_fp_dentro_real_0 = 100 * fp / total_real_0

pct_tp_dentro_real_1 = 100 * tp / total_real_1
pct_fn_dentro_real_1 = 100 * fn / total_real_1

print("====================================")
print(" MATRIZ DE CONFUSIÓN CON PORCENTAJES")
print("====================================")

print("TOTAL TEST:", total)

print("\n--- Porcentaje sobre todo el test ---")
print("TP - Default bien detectado       : {} ({:.2f}%)".format(tp, pct_tp_global))
print("TN - No default bien detectado    : {} ({:.2f}%)".format(tn, pct_tn_global))
print("FP - Falso default                : {} ({:.2f}%)".format(fp, pct_fp_global))
print("FN - Default no detectado         : {} ({:.2f}%)".format(fn, pct_fn_global))

print("\n--- Porcentaje dentro de cada clase real ---")
print("De los NO default reales:")
print("TN - Correctamente clasificados como no default : {} ({:.2f}%)".format(tn, pct_tn_dentro_real_0))
print("FP - Clasificados erróneamente como default     : {} ({:.2f}%)".format(fp, pct_fp_dentro_real_0))

print("\nDe los default reales:")
print("TP - Correctamente detectados como default      : {} ({:.2f}%)".format(tp, pct_tp_dentro_real_1))
print("FN - No detectados como default                 : {} ({:.2f}%)".format(fn, pct_fn_dentro_real_1))

print("====================================")

In [ ]:
# ============================================================
# CHUNK — Guardar métricas ya calculadas en refined/silver
# ============================================================

REFINED_PATH = "s3://hcdr/refined/"
MODEL_NAME = "RandomForestClassifier"

# ------------------------------------------------------------
# 1. Guardar métricas generales
# ------------------------------------------------------------

metrics_rows = [
    (MODEL_NAME, "accuracy", float(accuracy), "Porcentaje total de aciertos del modelo"),
    (MODEL_NAME, "precision", float(precision), "De los clientes predichos como default, cuántos realmente eran default"),
    (MODEL_NAME, "recall", float(recall), "De los clientes default reales, cuántos detectó el modelo"),
    (MODEL_NAME, "f1_score", float(f1_score), "Balance entre precision y recall"),
    (MODEL_NAME, "auc", float(auc), "Capacidad general del modelo para separar default y no default")
]

metrics_df = spark.createDataFrame(
    metrics_rows,
    ["modelo", "metrica", "valor", "descripcion_negocio"]
)

metrics_df.show(truncate=False)

metrics_df.write.mode("overwrite").parquet(
    REFINED_PATH + "model_metrics_random_forest/"
)

print("✅ Métricas guardadas en refined:")
print(REFINED_PATH + "model_metrics_random_forest/")

In [ ]:
# ============================================================
# CHUNK — Guardar predicciones en refined/silver
# ============================================================

REFINED_PATH = "s3://hcdr/refined/"

predictions_output = predictions.select(
    "sk_id_curr",
    "target",
    "label",
    "prediction",
    "probability"
)

predictions_output.show(20, truncate=False)

predictions_output.write.mode("overwrite").parquet(
    REFINED_PATH + "model_predictions_random_forest/"
)

print("✅ Predicciones guardadas en refined:")
print(REFINED_PATH + "model_predictions_random_forest/")

In [ ]:
# ============================================================
# CHUNK — Ver muestra de métricas guardadas en refined
# ============================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

REFINED_PATH = "s3://hcdr/refined/"

# Leer métricas guardadas
metricas_guardadas = spark.read.parquet(
    REFINED_PATH + "model_metrics_random_forest/"
)

print("Muestra de métricas guardadas en refined:")
metricas_guardadas.show(truncate=False)

In [ ]:
# ============================================================
# CHUNK — Ver muestra de matriz de confusión guardada en refined
# ============================================================

confusion_guardada = spark.read.parquet(
    REFINED_PATH + "confusion_matrix_random_forest/"
)

print("Muestra de matriz de confusión guardada en refined:")
confusion_guardada.show(truncate=False)

In [ ]:
# ============================================================
# CHUNK — Guardar matriz de confusión ya calculada
# ============================================================

total = tp + tn + fp + fn

total_real_0 = tn + fp
total_real_1 = tp + fn

confusion_rows = [
    (
        MODEL_NAME,
        "TP",
        "Default real",
        "Predijo default",
        int(tp),
        float(100 * tp / total),
        float(100 * tp / total_real_1 if total_real_1 > 0 else 0),
        "Clientes con default correctamente detectados"
    ),
    (
        MODEL_NAME,
        "TN",
        "No default real",
        "Predijo no default",
        int(tn),
        float(100 * tn / total),
        float(100 * tn / total_real_0 if total_real_0 > 0 else 0),
        "Clientes sin default correctamente clasificados"
    ),
    (
        MODEL_NAME,
        "FP",
        "No default real",
        "Predijo default",
        int(fp),
        float(100 * fp / total),
        float(100 * fp / total_real_0 if total_real_0 > 0 else 0),
        "Clientes sin default marcados erróneamente como default"
    ),
    (
        MODEL_NAME,
        "FN",
        "Default real",
        "Predijo no default",
        int(fn),
        float(100 * fn / total),
        float(100 * fn / total_real_1 if total_real_1 > 0 else 0),
        "Clientes con default que el modelo no detectó"
    )
]

confusion_df = spark.createDataFrame(
    confusion_rows,
    [
        "modelo",
        "tipo_resultado",
        "clase_real",
        "clase_predicha",
        "cantidad",
        "pct_sobre_total_test",
        "pct_dentro_clase_real",
        "interpretacion_negocio"
    ]
)

confusion_df.show(truncate=False)

confusion_df.write.mode("overwrite").parquet(
    REFINED_PATH + "confusion_matrix_random_forest/"
)

print("✅ Matriz de confusión guardada en refined:")
print(REFINED_PATH + "confusion_matrix_random_forest/")